# Regression Trees and Ensembles — Exercise

**Dataset:** Car fuel consumption in litres per 100 km.

Read [regression_trees_ensembles_concept.ipynb](regression_trees_ensembles_concept.ipynb) first. Run cells from top to bottom. Data preparation and plotting code are provided. Replace each `None` marked with a `# TODO` comment with the expression requested above the cell. These cells are incomplete until you fill the gap. Answer interpretation questions in one or two sentences. Check your work with [regression_trees_ensembles_solution.ipynb](regression_trees_ensembles_solution.ipynb).

## Libraries and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.inspection import permutation_importance

## 1. Compute a split

Left targets are 10,12,14; right targets are 30,34. Replace the gap with `np.sum((left - left.mean()) ** 2)`. Compare this split after 14 with the supplied alternative after 12. Which has smaller SSE? Why does each leaf predict its mean?

In [ ]:
left = np.array([10, 12, 14])
right = np.array([30, 34])
# TODO: Replace None with the expression requested above.
sse_left = None
sse_right = np.sum((right - right.mean()) ** 2)
print('Predictions:', left.mean(), right.mean())
print('Total split SSE:', sse_left + sse_right)
# Compare splitting after 12 instead of after 14.
alternative_left = np.array([10, 12])
alternative_right = np.array([14, 30, 34])
alternative_sse = (np.sum((alternative_left - alternative_left.mean()) ** 2)
                   + np.sum((alternative_right - alternative_right.mean()) ** 2))
print('Alternative split SSE:', alternative_sse)

> ✏️ **Your interpretation**
>
> *Double-click this cell and replace this text with your answer.*

## 2. Prepare car data

The preparation is provided. What does `combined` measure?

In [ ]:
cars = pd.read_csv('../00_Data/car_fuel_consumption_2025.csv')
cars = cars.dropna(subset=['engine_size', 'cylinders', 'combined'])
print(cars[['engine_size', 'cylinders', 'combined']].head())
X = cars[['engine_size', 'cylinders']]
y = cars['combined']  # litres per 100 km
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print('Training rows:', len(X_train), 'Test rows:', len(X_test))

> ✏️ **Your interpretation**
>
> *Double-click this cell and replace this text with your answer.*

## 3. Fit and read a small tree

Replace the gap with `DecisionTreeRegressor(max_depth=2, random_state=42)`. Follow the rules for engine size 2 and 4 cylinders. The `value` in a leaf is its predicted consumption.

In [ ]:
# TODO: Replace None with the expression requested above.
tree = None
tree.fit(X_train, y_train)
plt.figure(figsize=(10, 5))
plot_tree(tree, feature_names=list(X.columns), filled=True, rounded=True, precision=2)
plt.show()
new_car = pd.DataFrame({'engine_size': [2.0], 'cylinders': [4]})
print('Prediction:', tree.predict(new_car))

> ✏️ **Your interpretation**
>
> *Double-click this cell and replace this text with your answer.*

## 4. See what pruning changes

Two settings are provided as a demonstration. Compare the number of leaves and training errors. Which tree is easier to describe? We use CV in step 6 for the fixed candidate comparison.

In [ ]:
for alpha in [0.0, 0.05]:
    pruned = DecisionTreeRegressor(ccp_alpha=alpha, random_state=42)
    pruned.fit(X_train, y_train)
    print('alpha:', alpha, 'leaves:', pruned.get_n_leaves(),
          'training RMSE:', round(np.sqrt(mean_squared_error(
              y_train, pruned.predict(X_train))), 2))

> ✏️ **Your interpretation**
>
> *Double-click this cell and replace this text with your answer.*

## 5. Train three ensembles

The ensemble code is provided. Explain the difference between bagging/forest and boosting. What does the forest OOB R² describe?

In [ ]:
bagging = BaggingRegressor(n_estimators=50, random_state=42)
forest = RandomForestRegressor(n_estimators=100, max_features=1,
                               oob_score=True, random_state=42)
boosting = GradientBoostingRegressor(n_estimators=100, max_depth=2,
                                     learning_rate=0.1, random_state=42)
bagging.fit(X_train, y_train)
forest.fit(X_train, y_train)
boosting.fit(X_train, y_train)
print('Bagging:', bagging.predict(new_car))
print('Forest:', forest.predict(new_car))
print('Boosting:', boosting.predict(new_car))
print('Forest OOB R²:', round(forest.oob_score_, 3))

> ✏️ **Your interpretation**
>
> *Double-click this cell and replace this text with your answer.*

## 6. Compare fixed candidates using CV

This small comparison is fully supplied. Which candidate has the lowest CV RMSE? The best CV candidate is then evaluated on the test set.

In [ ]:
models = {
    'Linear': LinearRegression(),
    'Pruned tree': DecisionTreeRegressor(ccp_alpha=0.05, random_state=42),
    'Bagging': bagging,
    'Forest': forest,
    'Boosting': boosting,
}
cv_rmse = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5,
                             scoring='neg_mean_squared_error')
    cv_rmse[name] = np.sqrt(-scores).mean()
print(pd.Series(cv_rmse, name='CV RMSE').round(2))
best_name = min(cv_rmse, key=cv_rmse.get)
best_model = models[best_name]
best_model.fit(X_train, y_train)
print('Chosen model:', best_name)
print('Final test RMSE:', round(np.sqrt(mean_squared_error(
    y_test, best_model.predict(X_test))), 2))

> ✏️ **Your interpretation**
>
> *Double-click this cell and replace this text with your answer.*

## 7. Interpret feature importance

Provided code shuffles one forest input at a time on training data. Which input causes a larger loss in R²? This illustration uses training rows; training importance can be optimistic and does not establish causality.

In [ ]:
importance = permutation_importance(forest, X_train, y_train,
                                    n_repeats=5, random_state=42)
print(pd.Series(importance.importances_mean,
                index=X.columns, name='Decrease in R²').round(3))

> ✏️ **Your interpretation**
>
> *Double-click this cell and replace this text with your answer.*

### Jupyter notebook — footer info

Include this information when submitting your notebook.

In [ ]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))